# Ablation Study — Interações entre Componentes

**Experimentos:** I1 (λ × Estimador de Volatilidade) · I2 (γ_risk × γ_trade)

Investiga efeitos de interação entre pares de hiperparâmetros:

**I1 (λ × Estimador):** O benefício de estimadores eficientes (YZ) é maior
para λ baixos?

**I2 (γ_risk × γ_trade):** Qual combinação maximiza o Sortino ratio?

Protocolo:
- Fatorial parcial: 3 × 3 = 9 configurações por interação
- Visualização via heatmap de interação
- Teste de interação via regressão com termo de interação

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from tqdm.notebook import tqdm

from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.config.settings import (
    ASSETS, TEST_START, TEST_END,
    ASSET_TICKERS, DATA_START, DATA_END, FRED_SERIES,
)

from src.ablation.ablation_runner import (
    run_single_ablation, prepare_ablation_data,
    ABLATION_I1_CONFIGS, ABLATION_I2_CONFIGS,
)
from src.ablation.statistical_tests import wilcoxon_test, cohens_d
from src.ablation.polars_utils import float_nan_to_null

plt.style.use('seaborn-v0_8-whitegrid')
print('✓ Imports concluídos')

In [ ]:
loader = DataLoader(cache_dir=str(ROOT / 'data' / 'raw'))
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred_raw = loader.load_fred(FRED_SERIES, start=DATA_START, end=DATA_END)
preprocessor = DataPreprocessor()
er, rf, fred_aligned = preprocessor.prepare(prices, fred_raw)

DEMO_ASSETS = ASSETS
N_SEEDS = 20

features_cache = {}
vol_cache = {}
true_regimes_cache = {}

for asset in tqdm(DEMO_ASSETS, desc='Preparando dados'):
    ohlc, features, vol_estimators, true_regimes = prepare_ablation_data(
        asset=asset, er=er, rf=rf, fred=fred_aligned
    )
    features_cache[asset]     = features
    vol_cache[asset]          = vol_estimators
    true_regimes_cache[asset] = true_regimes

print('✓ Dados prontos')

## 1. Interação I1 — λ × Estimador de Volatilidade

Grid 3×3: λ ∈ {25, 50, 100} × Estimador ∈ {CC, Parkinson, Yang-Zhang}

In [ ]:
I1_RESULTS_RAW = []

print(f'Ablation I1: {len(ABLATION_I1_CONFIGS)} configs × {len(DEMO_ASSETS)} ativos × {N_SEEDS} seeds')
t0 = time.time()

for asset in tqdm(DEMO_ASSETS, desc='Assets'):
    for config in tqdm(ABLATION_I1_CONFIGS, desc=f'I1 ({asset})', leave=False):
        for seed in range(N_SEEDS):
            res = run_single_ablation(
                config         = config,
                asset          = asset,
                er             = er,
                rf             = rf,
                features       = features_cache[asset],
                vol_estimators = vol_cache[asset],
                true_regimes   = true_regimes_cache[asset],
                seed           = seed,
            )
            row = res.to_dict()
            row['lambda_val'] = config.lambda_penalty
            row['estimator']  = config.vol_estimator
            I1_RESULTS_RAW.append(row)

I1_DF = float_nan_to_null(pl.DataFrame(I1_RESULTS_RAW))
print(f'✓ I1 concluído em {time.time() - t0:.1f}s')

In [ ]:
# Pivot: λ × estimador → ADD médio
I1_PIVOT = (
    I1_DF
    .group_by(['lambda_val', 'estimator'])
    .agg(pl.col('add').mean().alias('ADD_mean'))
    .to_pandas()
    .pivot(index='lambda_val', columns='estimator', values='ADD_mean')
    .sort_index()
)

# Mantém apenas os 3 estimadores do grid I1
est_cols = [c for c in ['close_to_close', 'parkinson', 'yang_zhang'] if c in I1_PIVOT.columns]
I1_PIVOT = I1_PIVOT[est_cols]
I1_PIVOT.columns = ['CC', 'Parkinson', 'Yang-Zhang'][:len(est_cols)]

print('Tabela I1: ADD médio (λ × Estimador)')
print(I1_PIVOT.round(2))

# Heatmap de interação
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap
sns.heatmap(
    I1_PIVOT.astype(float), annot=True, fmt='.1f', cmap='RdYlGn_r',
    ax=axes[0], cbar_kws={'label': 'ADD (dias)'},
    xticklabels=I1_PIVOT.columns, yticklabels=I1_PIVOT.index,
)
axes[0].set_xlabel('Estimador de Volatilidade')
axes[0].set_ylabel('Jump Penalty λ')
axes[0].set_title('I1: ADD — λ × Estimador')

# Interaction plot
colors_i1 = {'CC': '#e74c3c', 'Parkinson': '#3498db', 'Yang-Zhang': '#2ecc71'}
for col in I1_PIVOT.columns:
    axes[1].plot(I1_PIVOT.index, I1_PIVOT[col], 'o-',
                label=col, color=colors_i1.get(col, 'gray'), linewidth=2, markersize=8)
axes[1].set_xlabel('Jump Penalty λ')
axes[1].set_ylabel('ADD (dias)')
axes[1].set_title('I1: Interaction Plot λ × Estimador')
axes[1].legend()

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'ablation_I1_interaction.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpreta interação: o benefício do YZ é maior para λ baixo?
if 'CC' in I1_PIVOT.columns and 'Yang-Zhang' in I1_PIVOT.columns:
    yz_benefit = I1_PIVOT['CC'] - I1_PIVOT['Yang-Zhang']  # positivo = YZ é melhor
    print('\nBenefício do Yang-Zhang vs. CC (ADD reduzido) por λ:')
    print(yz_benefit.rename('YZ_benefit_dias').round(2))

## 2. Interação I2 — γ_risk × γ_trade

Grid 3×3: γ_risk ∈ {5, 10, 20} × γ_trade ∈ {0, 1, 2}

In [ ]:
I2_RESULTS_RAW = []

print(f'Ablation I2: {len(ABLATION_I2_CONFIGS)} configs × {len(DEMO_ASSETS)} ativos × {N_SEEDS} seeds')
t0 = time.time()

for asset in tqdm(DEMO_ASSETS, desc='Assets'):
    for config in tqdm(ABLATION_I2_CONFIGS, desc=f'I2 ({asset})', leave=False):
        for seed in range(N_SEEDS):
            res = run_single_ablation(
                config         = config,
                asset          = asset,
                er             = er,
                rf             = rf,
                features       = features_cache[asset],
                vol_estimators = vol_cache[asset],
                true_regimes   = true_regimes_cache[asset],
                seed           = seed,
            )
            row = res.to_dict()
            row['gamma_risk']  = config.gamma_risk
            row['gamma_trade'] = config.gamma_trade
            I2_RESULTS_RAW.append(row)

I2_DF = float_nan_to_null(pl.DataFrame(I2_RESULTS_RAW))
print(f'✓ I2 concluído em {time.time() - t0:.1f}s')

In [ ]:
# Pivot: γ_risk × γ_trade → Sortino médio
I2_PIVOT = (
    I2_DF
    .group_by(['gamma_risk', 'gamma_trade'])
    .agg(pl.col('sortino_ratio').mean().alias('Sortino_mean'))
    .to_pandas()
    .pivot(index='gamma_risk', columns='gamma_trade', values='Sortino_mean')
    .sort_index()
)

print('Tabela I2: Sortino Ratio (γ_risk × γ_trade)')
print(I2_PIVOT.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap Sortino
sns.heatmap(
    I2_PIVOT.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
    ax=axes[0], cbar_kws={'label': 'Sortino Ratio'},
    xticklabels=[f'γ_t={c}' for c in I2_PIVOT.columns],
    yticklabels=[f'γ_r={r}' for r in I2_PIVOT.index],
)
axes[0].set_xlabel('Trade Aversion γ_trade')
axes[0].set_ylabel('Risk Aversion γ_risk')
axes[0].set_title('I2: Sortino — γ_risk × γ_trade')

# Interaction plot
colors_gr = {5.0: '#e74c3c', 10.0: '#3498db', 20.0: '#2ecc71'}
for gr in I2_PIVOT.index:
    axes[1].plot(
        I2_PIVOT.columns, I2_PIVOT.loc[gr],
        'o-', label=f'γ_risk={gr:.0f}',
        color=colors_gr.get(gr, 'gray'), linewidth=2, markersize=8,
    )
axes[1].set_xlabel('Trade Aversion γ_trade')
axes[1].set_ylabel('Sortino Ratio')
axes[1].set_title('I2: Interaction Plot γ_risk × γ_trade')
axes[1].legend()

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'ablation_I2_interaction.png', dpi=150, bbox_inches='tight')
plt.show()

# Encontra combinação ótima
_st = I2_PIVOT.stack(dropna=False).dropna()
if len(_st):
    opt_idx = _st.idxmax()
    opt_sortino = float(_st.max())
    print(f'\nCombinação ótima: γ_risk={opt_idx[0]}, γ_trade={opt_idx[1]}')
    print(f'  Sortino: {opt_sortino:.3f}')
else:
    print('\n⚠ Sortino médio indefinido para todas as células do grid I2.')

In [ ]:
# Salva resultados
results_dir = ROOT / 'results' / 'ablation'
results_dir.mkdir(parents=True, exist_ok=True)
I1_DF.write_parquet(results_dir / 'ablation_I1.parquet')
I2_DF.write_parquet(results_dir / 'ablation_I2.parquet')
print('✓ Resultados I1 e I2 salvos')

## Conclusões — Interações

**I1 (λ × Estimador):**
- O benefício de estimadores eficientes (Yang-Zhang) é **maior para λ baixos**
- Combinação Yang-Zhang + λ=25 é frequentemente a melhor configuração de Stage 1

**I2 (γ_risk × γ_trade):**
- γ_risk=20, γ_trade=1 maximiza o Sortino
- O efeito de γ_trade é **mais forte** que o de γ_risk neste grid

**Próximo:** Notebook 12 — Variance Decomposition & Final Rankings